# C01 · Registers, masks and volatile

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/01_c_bare_metal/C01_registers_masks_and_volatile.ipynb)

**Track:** C and bare-metal · **Skill axis (sheet):** C and bare-metal · **Time:** about 50 min

**Skill Hub concepts:** Bit mask and shift; Read-modify-write; Memory-mapped registers; volatile; Atomic set and reset registers

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("C01")

## Map: one register, five operations

A peripheral register is a 32-bit word at a fixed address. C reaches it through a pointer to a `volatile` struct whose
member offsets match the reference manual.

| Operation | Expression | Read-modify-write? | Safe if an interrupt touches the same register? |
|---|---|---|---|
| Set bit n | `R \|= (1u << n)` | yes | no |
| Clear bit n | `R &= ~(1u << n)` | yes | no |
| Read field (pos, width) | `(R >> pos) & mask(width)` | no | yes |
| Write field | `R = (R & ~(mask << pos)) \| ((v & mask) << pos)` | yes | no |
| Set or reset pin n through BSRR | `BSRR = 1u << n` or `BSRR = 1u << (n + 16)` | **no, one write** | **yes** |

$$\text{mask}(w) = 2^{w} - 1 \quad (w < 32), \qquad \text{mask}(32) = \texttt{0xFFFFFFFF}$$

STM32G4 GPIO port memory map (base GPIOA = `0x4800_0000`, each port is `0x400` apart):

| Offset | Register | Use |
|---|---|---|
| 0x00 | MODER | 2 bits per pin: 00 input, 01 output, 10 alternate, 11 analog |
| 0x04 | OTYPER | push-pull or open-drain |
| 0x08 | OSPEEDR | slew rate |
| 0x0C | PUPDR | pull-up / pull-down |
| 0x10 | IDR | input data (read only) |
| 0x14 | ODR | output data |
| 0x18 | BSRR | bit set (low 16) / bit reset (high 16), write only |
| 0x1C | LCKR | lock |
| 0x20, 0x24 | AFR[0], AFR[1] | alternate function, 4 bits per pin |
| 0x28 | BRR | bit reset |

## Task 1: the register struct and field helpers
Write `gpio.h` (struct with the exact offsets above) and `regs.c`. The hidden harness checks every offset with
`offsetof`, runs your helpers on a fake GPIO block in RAM, and compiles with the undefined-behavior sanitizer on
(so `1u << 32` fails).

In [ ]:
%%writefile gpio.h
#include <stdint.h>
typedef struct {
/* TODO: members in order, each volatile uint32_t, so offsets match the table */
    volatile uint32_t MODER;
} GPIO_TypeDef;

uint32_t field_read(uint32_t reg, unsigned pos, unsigned width);
uint32_t field_write(uint32_t reg, unsigned pos, unsigned width, uint32_t val);
void gpio_set_mode(GPIO_TypeDef *g, unsigned pin, unsigned mode);   /* read-modify-write MODER */
void gpio_write(GPIO_TypeDef *g, unsigned pin, int level);         /* one write to BSRR, never touch ODR */

In [ ]:
%%writefile regs.c
#include "gpio.h"

static uint32_t mask(unsigned width) {
    return 0; /* TODO: careful with width 32 */
}

uint32_t field_read(uint32_t reg, unsigned pos, unsigned width) {
    return 0; /* TODO */
}

uint32_t field_write(uint32_t reg, unsigned pos, unsigned width, uint32_t val) {
    (void)pos; (void)width; (void)val;
    return reg; /* TODO: clear the field, then insert val (masked) */
}

void gpio_set_mode(GPIO_TypeDef *g, unsigned pin, unsigned mode) {
    (void)g; (void)pin; (void)mode; /* TODO */
}

void gpio_write(GPIO_TypeDef *g, unsigned pin, int level) {
    (void)g; (void)pin; (void)level; /* TODO */
}

In [ ]:
lab.check("regs")

## Task 2: why `volatile` exists
`wait_ready` spins until bit 0 of a status word becomes 1. The harness builds it with `-O2`, starts a second
thread that sets the bit after 50 ms, and fails if your loop never returns. Without `volatile` the compiler may
load the word once and spin forever on a register copy.

(Sharing a plain variable between threads is still a data race; real thread or core sync uses atomics, which is lab C07.
This test stands in for a hardware status register, where `volatile` is exactly right.)

In [ ]:
%%writefile wait.c
#include <stdint.h>
void wait_ready(uint32_t *status) {
    while ((*status & 1u) == 0u) { }   /* TODO: make this survive -O2 */
}

In [ ]:
lab.check("volatile")

See it for yourself: the same loop with and without `volatile`, compiled at `-O2`. Find the loop label and check
whether the load (`mov (%rdi), ...`) sits inside it.

In [ ]:
from roboprep import clab
open("spin.c", "w").write("""#include <stdint.h>
void spin_plain(uint32_t *s) { while ((*s & 1u) == 0u) { } }
void spin_volatile(volatile uint32_t *s) { while ((*s & 1u) == 0u) { } }
""")
asm = clab.sh("gcc -O2 -S -fno-asynchronous-unwind-tables -fcf-protection=none -o - spin.c").stdout
print("\n".join(l for l in asm.splitlines() if not l.startswith("\t.") and not l[:1].isdigit()))

## Checkpoint: Math
Give exact integers (Python hex literals are fine).

**M1 (Apply).** GPIOA `MODER` resets to `0xABFFFFFF`. Value after configuring PA5 as a general-purpose output (mode 01)?

**M2 (Apply).** One `BSRR` write that sets pin 3 and resets pin 9 of the same port.

**M3 (Analyze).** Absolute address of `GPIOC->BSRR`.

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Main code toggles PA1 with `GPIOA->ODR ^= (1u << 1)` while an interrupt drives PA2 the same way.
Why does switching both to `BSRR` fix the occasional lost update?
- a) BSRR is clocked faster than ODR
- b) a single write has no read-modify-write window for the interrupt to land in
- c) BSRR makes the pointer volatile automatically

**K2 (Evaluate).** Explain what `volatile` guarantees and what it does not, for (1) a status register polled in a loop
and (2) a 32-bit flag shared between `main` and an interrupt handler.

In [ ]:
lab.choose("K1", None)  # "a", "b" or "c"

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})  # rubric points you actually covered

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `c01.h`).
3. In `roboprep/checks/c01.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()